실습시작

In [1]:
from google.colab import drive
drive.mount('/content/drive')
#mkdir -p /content/drive/MyDrive/aiffel
import os
os.chdir('/content/drive/MyDrive/aiffel')

Mounted at /content/drive


In [2]:
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        print("입력 Shape:", src.size())

        embedded = self.embedding(src)
        print("Embedding Layer를 거친 Shape:", embedded.size())

        outputs, (h_0, c_0) = self.rnn(embedded)
        print("LSTM Layer의 Output Shape:", outputs.size())
        print("LSTM Layer의 Hidden State Shape:", h_0.size())
        print("LSTM Layer의 Cell State Shape:", c_0.size())

        return outputs, h_0, c_0

print("슝~")

슝~


In [3]:
vocab_size = 30000
emb_size = 256
lstm_size = 512
batch_size = 1
sample_seq_len = 3

print("Vocab Size: {0}".format(vocab_size))
print("Embedding Size: {0}".format(emb_size))
print("LSTM Size: {0}".format(lstm_size))
print("Batch Size: {0}".format(batch_size))
print("Sample Sequence Length: {0}\n".format(sample_seq_len))

Vocab Size: 30000
Embedding Size: 256
LSTM Size: 512
Batch Size: 1
Sample Sequence Length: 3



In [4]:
import torch

encoder = Encoder(vocab_size, emb_size, lstm_size)
sample_input = torch.randint(0, vocab_size, (batch_size, sample_seq_len))

sample_output, hidden, cell = encoder(sample_input)

입력 Shape: torch.Size([1, 3])
Embedding Layer를 거친 Shape: torch.Size([1, 3, 256])
LSTM Layer의 Output Shape: torch.Size([1, 3, 512])
LSTM Layer의 Hidden State Shape: torch.Size([1, 1, 512])
LSTM Layer의 Cell State Shape: torch.Size([1, 1, 512])


In [5]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden, cell, context):
        print("입력 Shape:", x.size())

        embedded = self.embedding(x)
        print("Embedding Layer를 거친 Shape:", embedded.size())

        embedded = torch.cat((embedded, context), dim=2)
        print("Context Vector가 더해진 Shape:", embedded.size())

        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        print("LSTM Layer의 Output Shape:", output.size())

        output = self.fc(output)
        print("Decoder 최종 Output Shape:", output.size())

        return output, hidden, cell

In [6]:
print("Vocab Size: {0}".format(vocab_size))
print("Embedding Size: {0}".format(emb_size))
print("LSTM Size: {0}".format(lstm_size))
print("Batch Size: {0}".format(batch_size))
print("Sample Sequence Length: {0}\n".format(sample_seq_len))

Vocab Size: 30000
Embedding Size: 256
LSTM Size: 512
Batch Size: 1
Sample Sequence Length: 3



In [7]:
decoder_input = torch.randint(0, vocab_size, (batch_size, sample_seq_len))
decoder = Decoder(vocab_size, emb_size, lstm_size)

# 인코더 최종 상태를 디코더 매 스텝에 같은 값으로 붙임
context = hidden.transpose(0, 1).expand(-1, sample_seq_len, -1)

dec_output, hidden, cell = decoder(decoder_input, hidden, cell, context)

입력 Shape: torch.Size([1, 3])
Embedding Layer를 거친 Shape: torch.Size([1, 3, 256])
Context Vector가 더해진 Shape: torch.Size([1, 3, 768])
LSTM Layer의 Output Shape: torch.Size([1, 3, 512])
Decoder 최종 Output Shape: torch.Size([1, 3, 30000])


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim, units):
        super(BahdanauAttention, self).__init__()
        self.W_decoder = nn.Linear(hidden_dim, units)  # Decoder hidden state -> units
        self.W_encoder = nn.Linear(hidden_dim, units)  # Encoder hidden state -> units
        self.W_combine = nn.Linear(units, 1)           # Alignment score -> scalar weight

    def forward(self, H_encoder, H_decoder):
        print("[ H_encoder ] Shape:", H_encoder.shape)          # (batch, seq_len, hidden_dim)

        W_enc = self.W_encoder(H_encoder)                       # (batch, seq_len, units)
        print("[ W_encoder X H_encoder ] Shape:", W_enc.shape)

        print("\n[ H_decoder ] Shape:", H_decoder.shape)        # (batch, hidden_dim)
        W_dec = self.W_decoder(H_decoder.unsqueeze(1))          # (batch, 1, units)
        print("[ W_decoder X H_decoder ] Shape:", W_dec.shape)

        score = self.W_combine(torch.tanh(W_dec + W_enc))       # (batch, seq_len, 1)
        print("[ Score_alignment ] Shape:", score.shape)

        attention_weights = F.softmax(score, dim=1)             # (batch, seq_len, 1)
        print("\n최종 Weight:\n", attention_weights.squeeze(-1).detach().numpy())

        # 가중합은 사영(W_encoder)하기 전의 원본 인코더 상태에 적용한다
        context_vector = torch.sum(attention_weights * H_encoder, dim=1)   # (batch, hidden_dim)
        print("\n[ Context Vector ] Shape:", context_vector.shape)

        return context_vector, attention_weights

# 설정
hidden_dim = 512
W_size = 100
print(f"Hidden State를 {W_size}차원으로 Mapping\n")

# 모델 생성
attention = BahdanauAttention(hidden_dim, W_size)

# 입력 데이터 (배치 크기 = 1)
enc_state = torch.rand((1, 10, hidden_dim))  # (batch, seq_len, hidden_dim)
dec_state = torch.rand((1, hidden_dim))      # (batch, hidden_dim)

# 실행
_ = attention(enc_state, dec_state)

Hidden State를 100차원으로 Mapping

[ H_encoder ] Shape: torch.Size([1, 10, 512])
[ W_encoder X H_encoder ] Shape: torch.Size([1, 10, 100])

[ H_decoder ] Shape: torch.Size([1, 512])
[ W_decoder X H_decoder ] Shape: torch.Size([1, 1, 100])
[ Score_alignment ] Shape: torch.Size([1, 10, 1])

최종 Weight:
 [[0.09705343 0.09773125 0.10772386 0.09896756 0.10859531 0.09911867
  0.09407262 0.09798448 0.09854862 0.10020421]]

[ Context Vector ] Shape: torch.Size([1, 512])


In [9]:
class LuongAttention(nn.Module):
    def __init__(self, units):
        super(LuongAttention, self).__init__()
        self.W_combine = nn.Linear(units, units)  # Encoder hidden state 변환

    def forward(self, H_encoder, H_decoder):
        print("[ H_encoder ] Shape:", H_encoder.shape)  # (batch, seq_len, hidden_dim)

        WH = self.W_combine(H_encoder)  # (batch, seq_len, hidden_dim)
        print("[ W_encoder X H_encoder ] Shape:", WH.shape)

        H_decoder = H_decoder.unsqueeze(1)  # (batch, 1, hidden_dim)
        alignment = torch.bmm(WH, H_decoder.transpose(1, 2))  # (batch, seq_len, 1)
        print("[ Score_alignment ] Shape:", alignment.shape)

        attention_weights = F.softmax(alignment, dim=1)  # (batch, seq_len, 1)
        print("\n최종 Weight:\n", attention_weights.squeeze(-1).detach().numpy())

        attention_weights = attention_weights.squeeze(-1)  # (batch, seq_len)
        context_vector = torch.bmm(attention_weights.unsqueeze(1), H_encoder)  # (batch, 1, hidden_dim)
        context_vector = context_vector.squeeze(1)  # (batch, hidden_dim)

        return context_vector, attention_weights

# 설정
hidden_dim = 512
attention = LuongAttention(hidden_dim)

# 입력 데이터 (배치 크기 = 1)
enc_state = torch.rand((1, 10, hidden_dim))  # (batch, seq_len, hidden_dim)
dec_state = torch.rand((1, hidden_dim))  # (batch, hidden_dim)

# 실행
_ = attention(enc_state, dec_state)

[ H_encoder ] Shape: torch.Size([1, 10, 512])
[ W_encoder X H_encoder ] Shape: torch.Size([1, 10, 512])
[ Score_alignment ] Shape: torch.Size([1, 10, 1])

최종 Weight:
 [[0.06428779 0.31255862 0.04701682 0.3276464  0.00068305 0.00723265
  0.03496215 0.01714135 0.16946949 0.01900161]]
